In [1]:

!pip install pandas numpy scikit-learn tslearn prophet


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from tslearn.clustering import TimeSeriesKMeans
from prophet import Prophet

# -----------------------------
# 1. LOAD & PREPROCESS
# -----------------------------
df = pd.read_csv("energy_2023.csv")  # your dataset

# ID column
ids = df["ID"]

# Extract only the time-series values
ts_values = df.drop(columns=["ID"])

# Ensure columns are datelike
ts_values.columns = pd.to_datetime(ts_values.columns)

# Convert to (n_customers, n_days) numpy array
X = ts_values.values

# Optional: scaling improves clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data loaded. Shape:", X_scaled.shape)

# -----------------------------
# 2. TIME-SERIES CLUSTERING
# -----------------------------
n_clusters = 4  # baseline – tune later

# Using KMeans from tslearn for DTW or Euclidean distance
kmeans = TimeSeriesKMeans(
    n_clusters=n_clusters,
    metric="euclidean",  # can also use "dtw"
    random_state=42
)

clusters = kmeans.fit_predict(X_scaled)

cluster_df = pd.DataFrame({
    "ID": ids,
    "cluster": clusters
})

cluster_df.to_csv("clusters_output.csv", index=False)
print("Clustering done. Saved to clusters_output.csv")

# -----------------------------
# 3. FORECASTING BASELINE (PROPHET)
# -----------------------------
# You will need 2024 data for evaluation
df_2024 = pd.read_csv("energy_2024.csv")
df_2024.columns = [col if col == "ID" else pd.to_datetime(col) for col in df_2024.columns]

def prepare_prophet_format(series_2023):
    """Convert wide format row → Prophet long format."""
    s = pd.DataFrame({
        "ds": ts_values.columns,
        "y": series_2023.values
    })
    return s

def prophet_forecast(customer_id, horizon_days=30):
    """Train on 2023 data, forecast for 2024, and compare."""
    row_2023 = df[df["ID"] == customer_id].drop(columns=["ID"]).T
    row_2023.columns = ["y"]
    row_2023["ds"] = row_2023.index

    # 2024 truth
    row_2024 = df_2024[df_2024["ID"] == customer_id].drop(columns=["ID"]).T
    row_2024.columns = ["y"]
    row_2024["ds"] = row_2024.index

    # Fit model
    model = Prophet()
    model.fit(row_2023)

    # Forecast full 2024
    future = model.make_future_dataframe(periods=len(row_2024), freq="D")
    forecast = model.predict(future)

    # Extract forecast for evaluation window
    pred_2024 = forecast[forecast["ds"].isin(row_2024["ds"])]

    # Evaluation (MAE)
    mae = np.mean(np.abs(pred_2024["yhat"].values - row_2024["y"].values))

    return pred_2024, mae


# -----------------------------
# 4. EVALUATE A SAMPLE CUSTOMER
# -----------------------------
sample_id = ids.iloc[0]

pred_2024, mae = prophet_forecast(sample_id)
print(f"MAE for customer {sample_id}: {mae:.2f}")

pred_2024.to_csv(f"forecast_{sample_id}.csv", index=False)
print("Forecast saved!")

/home/fena/.local/lib/python3.10/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)
Importing plotly failed. Interactive plots will not work.


Data loaded. Shape: (17547, 365)
Clustering done. Saved to clusters_output.csv


/tmp/ipykernel_146555/496959091.py:79: FutureWarning: Setting an Index with object dtype into a DataFrame will stop inferring another dtype in a future version. Cast the Index explicitly before setting it into the DataFrame.
  row_2024["ds"] = row_2024.index
17:36:10 - cmdstanpy - INFO - Chain [1] start processing
17:36:10 - cmdstanpy - INFO - Chain [1] done processing


MAE for customer 22: 9.18
Forecast saved!
